FileFinder for AccessViz: This function finds a list of travel time matrix files based on a list of YKR ID values from a specified input data folder. The code works for different list lengths and different YKR ID values. If the YKR ID number does not exist in the input folder (and it’s subfolders), the tool will warn about this to the user but still continue running. The tool also informs the user about the execution process: tells the user what file is currently under process and how many files there are left (e.g. "Processing file travel_times_to_5797076.txt.. Progress: 3/25"). As output, FileFinder compiles a list of FilePaths for further processing. FileFinder can also print out a list of filepaths into a text file.

In [87]:
#import all the necessary modules
import pathlib
import re

#filefinder function

def filefinder(id_list, directory):
    """ A function that finds a list of travel time matrix files based on
    a list of YKR ID values and the input data directory.
    The code works for different list lengths and different YKR ID values.
    The function will return an empty list if an empty list is passed on.
    The function will warn about duplicates and skip them.
    The function will also warn about missing files.
    A progress report will also be shown for each search.
    Input : A list of YKR IDs and the input directory
    Return: A list of travel time matrix file paths. """

    string_id_list = [str(num) for num in id_list]

    #get a list of filepaths from the input directory
    filepaths = list( directory.rglob("travel_times_to_*.txt"))

    #get a list of filenames from the filepaths using pathlib
    filenames = [x.name for x in filepaths]

    #make a dictionary of ykr ids and corresponding filepaths
    files_dictionary = {(re.search(r"\d+", key)).group():value for key, value in zip(filenames, filepaths)}

    #empty list to store the required file paths
    files_out =[]

    #empty list to store id values to check for duplicates
    ids_list = []

    #loop to search through the files
    for i in range (len(string_id_list)):

        #conditional statemnet to check for duplicates
        if (string_id_list[i]) in ids_list:
            print(f"The ID: {string_id_list[i]} is a duplicate, so it will be skipped... Progress {i+1}/{len(string_id_list)}")
            continue
        else:

            #try and except to catch errors
            try:
                print(f"Processing file: {files_dictionary[string_id_list[i]]} ... Progress {i+1}/{len(string_id_list)}")
                files_out.append(files_dictionary[string_id_list[i]])

                #processed ids will only be added if the file exits
                ids_list.append(string_id_list[i])

            except KeyError:
                print(f"The ID: {string_id_list[i]} did not have a matching file, Check the YKR Id again... Progress {i+1}/{len(string_id_list)}")
    
            
        
    return(files_out)
        



In [95]:
#notebook path
NOTEBOOK_PATH = pathlib.Path()

#set the data directory
Data_Directory = NOTEBOOK_PATH / "data"

#input list of YKR IDs for analysis
id_list = ['5785640', '5785641', '5785642']

#output files
files = filefinder(id_list, Data_Directory)

print(files)

Processing file: data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785640.txt ... Progress 1/3
Processing file: data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785641.txt ... Progress 2/3
Processing file: data\HelsinkiRegion_TravelTimeMatrix2013\5785xxx\travel_times_to_ 5785642.txt ... Progress 3/3
[WindowsPath('data/HelsinkiRegion_TravelTimeMatrix2013/5785xxx/travel_times_to_ 5785640.txt'), WindowsPath('data/HelsinkiRegion_TravelTimeMatrix2013/5785xxx/travel_times_to_ 5785641.txt'), WindowsPath('data/HelsinkiRegion_TravelTimeMatrix2013/5785xxx/travel_times_to_ 5785642.txt')]


TableJoiner for AccessViz: This tool creates a spatial layer from the chosen Matrix text table (e.g. travel_times_to_5797076.txt) by joining the Matrix file with MetropAccess_YKR_grid Shapefile where from_id in Matrix file corresponds to YKR_ID in the Shapefile. The tool saves the result in the output-folder that user has defined. Output file format can be Shapefile or Geopackage. The files are named in a way that it is possible to identify the ID from the name (e.g. 5797076). The table joiing is applied to files that correspond to a list of selected YKR ID files (FileFinder handles finding the correct input files!).

In [108]:
#geopandas to read the YKR grid shapefile
import geopandas

#pandas to read the travel time matrix file
import pandas as pd

#set the output directory
Output_Directory = NOTEBOOK_PATH / "output"

#set the type of output files: :gpkg" or "shp"
output_type = "shp"

#read the file 
grid = geopandas.read_file(Data_Directory/"MetropAccess_YKR_grid/MetropAccess_YKR_grid")

grid.head()

#tablejoiner function
def tablejoiner(files, grid, Output_Directory, output_type):

    """ This tool creates a spatial layer from the chosen Matrix text table 
    (e.g. travel_times_to_5797076.txt) by joining the Matrix file with MetropAccess_YKR_grid Shapefile 
    where from_id in Matrix file corresponds to YKR_ID in the Shapefile. 
    The tool saves the result in the output-folder that user has defined. 
    Output file format can be Shapefile or Geopackage. 
    The files are named in a way that it is possible to identify the ID from the name (e.g. 5797076). T
    he table joiing is applied to files that correspond to a list of selected YKR ID files."""
    
    #empty list to store the output file paths
    output_file_paths = []

    #loop through the files returned from filefinder and merge them each with the grid file
    for file_path in files:

        #read the .text travel file
        travel_data = pd.read_csv(file_path, sep=";")
        #rename to make sure both this file and grid have same column names to merge
        travel_data = travel_data.rename(columns={"from_id": "YKR_ID"})
        #merge
        travel_data_grid = grid.merge(travel_data, on='YKR_ID', how='left')

        #get the file name using regex
        out_file_name = (re.search(r"\d+", file_path.name)).group()

        #choose the extension based on users choice
        if output_type == "shp":
            out_file_name_full = f"{out_file_name}.shp"
        else:
            out_file_name_full = f"{out_file_name}.gpkg"
            
        #save the file
        travel_data_grid.to_file(Output_Directory/out_file_name_full)
        #append the file paths to return later
        output_file_paths.append(Output_Directory/out_file_name_full)

    return(output_file_paths)

        
        
out_files = tablejoiner(files, grid, Output_Directory, output_type)

print(out_files)
        
    
    

[WindowsPath('output/5785640.shp'), WindowsPath('output/5785641.shp'), WindowsPath('output/5785642.shp')]
